# Mission 1 · WavLM-Large 우리 Train 성별 학습

고정 backbone 다층 특징 → Train 내부 dev 비교 → 조건부 상위 2층 적응 → 설정 잠금 → 전체 Train 재학습 → 기존 Internal Validation.
99%는 목표이며 보장하지 않습니다. GPU는 **T4 또는 L4**. T4는 FP32, L4는 BF16을 자동 선택합니다. 공식 Validation은 사용하지 않습니다.

`wavlm_gender_adaptation_t4_v2_bundle.zip`을 Drive `DDC-Colab/`에 업로드하세요. 기존 실험 GPU 작업을 먼저 종료하세요. 각 단계 출력 확인 후 다음 셀로 진행합니다. 오류가 나면 다음 단계로 넘어가지 않습니다.

## 1. Drive·bundle 검증 및 기존 경로 확인
기존 config 우선, 없으면 기존 Drive identity를 읽습니다. 새 코드/출력만 생성합니다.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, signal, subprocess, sys, time, zipfile

drive.mount('/content/drive')
BUNDLE = Path('/content/drive/MyDrive/DDC-Colab/wavlm_gender_adaptation_t4_v2_bundle.zip')
WORK = Path('/content/wavlm_gender_adaptation_v2')
EXPECTED_SHA256 = 'e0ff8703d2dfde1c8eb81d632c6cc936e57dd9f681027c87c9a0df3790db1d8b'
assert hashlib.sha256(BUNDLE.read_bytes()).hexdigest() == EXPECTED_SHA256, 'Notebook / ZIP version mismatch'
WORK.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    assert len(z.namelist()) == len(set(z.namelist()))
    assert all((WORK/n).resolve().is_relative_to(WORK.resolve()) for n in z.namelist())
    z.extractall(WORK)
old = Path('/content/gender_model_comparison/config.json')
previous = json.loads(old.read_text()) if old.exists() else None
SOURCE = Path(previous.get('output_dir', '/content/drive/MyDrive/DDC-Colab/gender_comparison_v1')) if previous else Path('/content/drive/MyDrive/DDC-Colab/gender_comparison_v1')
source_identity = json.loads((SOURCE/'identity.json').read_text())
DATA_ROOT = previous['data_root'] if previous else source_identity['data_root']
assert DATA_ROOT == source_identity['data_root']
assert Path(DATA_ROOT).is_dir()
# Query hardware before selecting precision or the isolated output identity.
import runpy
gpu_profile = runpy.run_path(str(WORK/'common.py'))['gpu_profile']
GPU_NAME = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).splitlines()[0].strip()
PROFILE = gpu_profile(GPU_NAME)
OUTPUT = SOURCE.parent/('wavlm_gender_adaptation_' + PROFILE['output_tag'])
assert OUTPUT != SOURCE
CONFIG = WORK/('config_' + PROFILE['output_tag'] + '.json')
# Predeclared 12s crop; benchmark first. Resource-driven alternative: 8/16s with NEW output identity, before development.
config = dict(data_root=DATA_ROOT, source_output=str(SOURCE), output_dir=str(OUTPUT), precision=PROFILE['precision'], expected_gpu=GPU_NAME, crop_seconds=12, accumulation=4)
if CONFIG.exists():
    assert json.loads(CONFIG.read_text()) == config, 'Use new output for changed configuration'
CONFIG.write_text(json.dumps(config, ensure_ascii=False, indent=2))
print('GPU:', GPU_NAME, 'precision:', PROFILE['precision'])
print('Original config:', previous, '\nSource:', SOURCE, '\nData:', DATA_ROOT, '\nNEW output:', OUTPUT)
print(json.loads((WORK/'design_manifest.json').read_text()))


## 2. 실제 환경 확인·별도 venv
정상 torch는 재설치하지 않습니다. torchaudio import/CUDA wheel이 불일치하면 중단합니다. 새 venv의 비 torch 패키지만 고정합니다. 모든 subprocess는 Stop 시 자식까지 종료합니다.


In [ ]:
import os
import sys
import json
import signal
import shutil
import subprocess
from pathlib import Path


def command(argv, env=None):
    def parent_death_signal():
        import ctypes

        parent = os.getppid()
        ctypes.CDLL(None).prctl(1, signal.SIGTERM)

        if os.getppid() != parent:
            os.kill(os.getpid(), signal.SIGTERM)

    proc = subprocess.Popen(
        list(map(str, argv)),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
        start_new_session=True,
        preexec_fn=parent_death_signal
    )

    try:
        for line in proc.stdout:
            print(line, end='', flush=True)

        if proc.wait():
            raise RuntimeError(
                f'Failed ({proc.returncode}); do not continue'
            )

    finally:
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGTERM)

            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL)
                proc.wait()


# 기존 gender_compare_env가 있으면 해당 Python 사용
SOURCE_PY = Path('/content/gender_compare_env/bin/python')

if not SOURCE_PY.exists():
    SOURCE_PY = Path(sys.executable)


# PyTorch / CUDA 환경 확인
probe = """
import torch
import torchaudio
import sys

print(sys.version)
print(torch.__version__)
print(torchaudio.__version__)
print(torch.version.cuda)

assert torch.cuda.is_available()

print(torch.cuda.get_device_name(0))

assert torch.__version__ == torchaudio.__version__, 'wheel mismatch'
"""

command([SOURCE_PY, '-c', probe])


# WavLM 전용 가상환경
ENV = Path('/content/wavlm_adaptation_env_v2')
PY = ENV / 'bin/python'


# 이전 실행에서 불완전하게 생성된 환경 제거
if ENV.exists():
    shutil.rmtree(ENV)


# ensurepip 오류를 피해서 가상환경 생성
command([
    SOURCE_PY,
    '-m',
    'venv',
    '--system-site-packages',
    '--without-pip',
    ENV
])


# 기존 환경의 site-packages 연결
source_sites = json.loads(
    subprocess.check_output(
        [
            SOURCE_PY,
            '-c',
            'import site,json; print(json.dumps(site.getsitepackages()))'
        ],
        text=True
    )
)

target_sites = json.loads(
    subprocess.check_output(
        [
            PY,
            '-c',
            'import site,json; print(json.dumps(site.getsitepackages()))'
        ],
        text=True
    )
)

local_site = next(
    Path(p)
    for p in target_sites
    if str(ENV) in p
)

(
    local_site / 'existing_torch_environment.pth'
).write_text(
    '\n'.join(source_sites) + '\n'
)


# 새 환경에서 GPU / PyTorch 확인
command([PY, '-c', probe])


# pip 확인
command([
    PY,
    '-m',
    'pip',
    '--version'
])


# requirements 설치
command([
    PY,
    '-m',
    'pip',
    'install',
    '-r',
    WORK / 'requirements.txt'
])


# 설치 이후에도 torch / torchaudio가 정상인지 재확인
command([PY, '-c', probe])


# Runner 실행 환경
RUN_ENV = dict(
    os.environ,
    HF_HOME='/content/gender_model_hf_cache',
    CUBLAS_WORKSPACE_CONFIG=':4096:8',
    TOKENIZERS_PARALLELISM='false'
)


def run(stage):
    command(
        [
            PY,
            '-u',
            WORK / 'runner.py',
            stage,
            '--config',
            CONFIG
        ],
        env=RUN_ENV
    )

!nvidia-smi

from pathlib import Path

lock_path = Path(
    '/content/drive/MyDrive/DDC-Colab/wavlm_gender_adaptation_l4_bf16_v2/.run.lock'
)

if lock_path.exists():
    print("기존 lock:", lock_path.read_text())
    lock_path.unlink()
    print(".run.lock 삭제 완료")
else:
    print(".run.lock 없음")


## 3. Preflight: 모델 revision·초기 가중치·RAM·VRAM·수치 검증
첫 다운로드에는 모델 로딩 시간이 별도로 듭니다. T4에서는 FP32 반복 실행의 finite·재현성을, L4에서는 FP32 대비 BF16 특징 상대오차를 확인합니다. 실패 시 새 FP32 identity로 다시 사전 검증합니다.


In [ ]:
run('preflight')


## 4. 실제 T4/L4 benchmark: 100 optimizer steps × frozen/top2 + 길이별 60통화 평가
WAV 읽기, resample, forward/backward, optimizer, Drive checkpoint 저장을 포함합니다. 워밍업 2 step 제외. 기본 frozen-only 선택과 별개로, 이 셀은 top2도 짧게 측정하고 학습된 가중치는 버립니다. 이후 출력의 epoch/전체 ETA·자원을 확인합니다. 6–20시간 같은 과거 계획 추정은 사용하지 않습니다. OOM이면 README 절차에 따라 새 출력에서 crop/precision을 변경하고 재측정합니다.


In [ ]:
run('benchmark')
print((OUTPUT/'benchmark.json').read_text())


## 5. Train caller/crop 화자 확인
Train metadata 위험 표본과 대조 표본의 전체 caller 및 실제 crop을 생성합니다. 상담원 혼입·화자 교체·라벨 불일치를 듣고 CSV를 작성하세요. 겹침 주석만으로 실제 화자가 틀렸다고 판단하지 않습니다. Frozen LR 단계는 진행할 수 있지만 적응 학습에는 검토 완료가 필요합니다.


In [ ]:
run('audit')
print('Review CSV:', OUTPUT/'audio_review/review.csv')


## 6. Frozen WavLM Train 특징 추출·4개 LR 후보 비교
22,388 Train만 추출. last-layer mean / 4-layer mean+std × C=0.1/1. dev는 고정 call split입니다. cache는 약 0.85 GiB, frame-level cache는 생성하지 않습니다.


In [ ]:
run('extract')
run('frozen-select')
print((OUTPUT/'frozen_selection.json').read_text())


## 7. 조건부 backbone 적응 비교
기본은 제한된 frozen-only 실행입니다. dev <99%이고 추가 비용이 타당하며 청취 검토를 끝냈으면 `RUN_ADAPTATION=True`로 변경합니다. Frozen neural control과 top2를 각각 최대 4 epochs, 같은 split·crop·head·seed로 비교합니다. 모두 끝난 다음에만 설정을 잠급니다. Internal Validation은 아직 접근하지 않습니다.


In [ ]:
RUN_ADAPTATION = False
if RUN_ADAPTATION:
    run('develop-frozen')
    run('develop-top2')
    print((OUTPUT/'develop_frozen/selection.json').read_text())
    print((OUTPUT/'develop_top2/selection.json').read_text())


## 8. 설정 잠금·전체 Train 재학습
잠금은 덮어쓰지 않습니다. 이후 dev 실험을 차단합니다. LR은 22,388개 전체로 fit. 선택된 top2는 초기 backbone/head부터 dev에서 정한 epoch 수로 전체 Train 재학습합니다. 선택된 최고 dev checkpoint를 그대로 최종 평가하지 않습니다.


In [ ]:
if not (OUTPUT/'selection.lock.json').exists():
    run('lock-adapted' if RUN_ADAPTATION else 'lock-frozen')
print((OUTPUT/'selection.lock.json').read_text())
run('final-fit')


## 9. 설정 고정 후 Internal Validation 5,597통화
모든 caller 보존. F/M argmax. 중간 오류 발생 시 같은 셀로 재개합니다. 완료 통화는 검증 후 건너뜁니다. 낮은 점수여도 Validation에 맞춰 설정을 변경하지 않습니다. 새로운 독립 test가 아닙니다.


In [ ]:
run('validation')
print((OUTPUT/'completion.json').read_text())


## 10. 결과 ZIP 다운로드·로컬 검증
모델 weight·원본 WAV·cache는 ZIP에 넣지 않습니다. Drive의 final checkpoint는 재개/재현을 위해 보존합니다.


In [ ]:
from google.colab import files
assert json.loads((OUTPUT/'completion.json').read_text())['status']=='complete'
files.download(str(OUTPUT/'wavlm_results.zip'))


## 중단·강제 종료 뒤 lock 점검
일반 Stop은 프로세스 그룹을 종료하고 lock을 해제합니다. SIGKILL/런타임 소실로 lock이 남으면 이전 VM/실행이 종료됐는지 확인하세요. 아래 함수는 owner token과 현재 GPU 상태를 검사합니다. Drive는 여러 VM에 대한 분산 lock을 보장하지 않습니다. 한 output에는 한 런타임만 사용합니다.


In [ ]:
def inspect_locks():
    for p in [Path('/content/.wavlm_gender_gpu.lock'), OUTPUT/'.run.lock']:
        if p.exists(): print(p, p.read_text())
    print(subprocess.check_output(['nvidia-smi'], text=True))
def release_stale_lock(path, owner_token, previous_runtime_stopped=False):
    assert previous_runtime_stopped, 'Confirm old process/runtime has terminated'
    path = Path(path)
    assert path in [Path('/content/.wavlm_gender_gpu.lock'), OUTPUT/'.run.lock']
    assert json.loads(path.read_text())['token'] == owner_token
    assert not subprocess.check_output(['nvidia-smi','--query-compute-apps=pid','--format=csv,noheader'],text=True).strip()
    path.unlink()
inspect_locks()
